# OSRT v6b — 200-step GRPO run (protocol notebook)

Purpose-built for **one declared run**, not a general tool. Settings are fixed
below and the interpretation is written down **before** it starts, so no reading
can be fitted to the result afterwards.

## What this run tests

Whether the reward's known misalignment is **fatal or merely suboptimal**. Two
40-step probes were too short to tell: one ended 3.5pp below the SFT-v4 soup
(p=0.104, crosses zero), the other 6.0pp below (p=0.030, excludes zero), and the
two did not differ from each other (p=0.365). 200 steps gives four panel points
and a trajectory *shape*, which nothing so far has.

## Declared readings — fixed now

| result at step 200 | conclusion |
|---|---|
| `acc_on` trending up, crossing **20.0%** | reward is good enough; verification becomes an optimisation, not a prerequisite |
| `acc_on` flat around **14–16%** | the reward is the ceiling. Strongest case for building verification, and it explains both probes |
| `acc_on` declining | actively harmful at scale. Stop and build verification |

**Primary:** absolute `acc_on` vs the soup's 20.0%, paired on the same items.
**Control:** `acc_off`. **Diagnostic only:** `delta` — it improves when `acc_off`
falls, which is how the wave-2 soup gained 4pp of delta while losing 3pp of
`acc_off`. Weight drift is diagnostic, never a verdict: wave 2 moved 0.145% and
lost 7.5pp.

**One seed, so no capability claim comes out of this** whatever it shows. That
needs a second seed plus the untouched `PROBLEM_OFFSET=200` confirmation set.

## What is NOT changed

`peak_lr` stays at 1.5e-6. Raising it would mean two changes at once, and at 200
steps the default already produces drift of the same order as wave 2's 0.145% —
which demonstrably moved behaviour. Step count is the single variable.

## Cost

200 × ~130s ≈ **7.2 hours**, so roughly three Colab sessions. Resume across
sessions now works: `/content/ckpt` dies with the VM, so the runner falls back to
scanning HF for the newest `<prefix>_step_N.pt`. Before that fix every new
session silently restarted from step 0 — three sessions would have produced
three ~65-step runs and never reached 200.

In [ ]:
# ── 1. Repo sync + secrets + THIS RUN's fixed settings ───────────────
import os, sys

BRANCH = "feat/grpo-v6b"
REPO = "https://github.com/CodeHalwell/OSRT-605M-A269M.git"
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/osrt
else:
    !cd /content/osrt && git remote set-url origin {REPO} \
        && git fetch -q --depth 1 origin {BRANCH} \
        && git checkout -q -B {BRANCH} FETCH_HEAD
!cd /content/osrt && git log --oneline -1 && git rev-parse --abbrev-ref HEAD

%pip -q install -U "transformers>=5.3.0" datasets tokenizers safetensors wandb huggingface_hub
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── FIXED for this run. Do not change mid-run. ───────────────────────
TOTAL_STEPS = 200
MAX_GEN_LEN = 1024    # length-ramp denominator; free thinking below 819 tokens
SEED        = 7       # seeds python + cpu torch + CUDA, logged in every ckpt
RUN_TAG     = "s7x200"  # keeps 20 interval saves clear of the 40-step artefacts
KL_ABORT    = 0.60   # see below
PEAK_LR     = 0.0     # 0 = config default (1.5e-6). ONE variable: step count.
HRA_LR      = 0.0

# KL_ABORT rationale: KL runs ~0.0024/step at beta=0.04, so 200 steps
# extrapolates to ~0.48. An abort at 0.25 would fire near step 100 and kill a run
# that is only drifting as predicted. 0.60 is a RUNAWAY detector, not a drift
# limiter — degradation is caught behaviourally by the panel evals instead.
for k, v in dict(TOTAL_STEPS=TOTAL_STEPS, MAX_GEN_LEN=MAX_GEN_LEN, SEED=SEED,
                 RUN_TAG=RUN_TAG, KL_ABORT=KL_ABORT, PEAK_LR=PEAK_LR,
                 HRA_LR=HRA_LR).items():
    os.environ[k] = str(v)

import torch
cap = torch.cuda.get_device_capability(0)
print(f"\n{torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]} | "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f}GB")
print(f"steps={TOTAL_STEPS} seed={SEED} tag={RUN_TAG} gen_len={MAX_GEN_LEN} "
      f"kl_abort={KL_ABORT} lr=cfg-default")

In [ ]:
# ── 2. Prompt set: PULL the validated file, never rebuild ────────────
import json, os, shutil

from huggingface_hub import hf_hub_download

OUT = "/content/grpo_prompts.jsonl"
if not os.path.exists(OUT):
    shutil.copy2(hf_hub_download("HallD/osrt-v6-ckpt", "data/grpo_prompts.jsonl",
                                 repo_type="model"), OUT)
rows = [json.loads(l) for l in open(OUT)]
bad = [i for i, r in enumerate(rows)
       if not str(r.get("answer","")).strip() or not str(r.get("question","")).strip()]
print(f"{len(rows)} prompts, {len(bad)} invalid gold")
assert not bad, f"invalid gold at lines {[i+1 for i in bad[:5]]} — re-pull"

In [ ]:
# ── 3. THE RUN — re-run this cell each session to RESUME ─────────────
# FOREGROUND, not nohup: a detached launch returns instantly, the notebook then
# looks IDLE, and Colab reclaims the VM out from under the background process.
#
# RESUMES AUTOMATICALLY. It prefers the newest local <prefix>_step_N.pt, then
# falls back to scanning HF (the VM's disk is gone at the start of each session),
# then the SFT-v4 soup. Watch the first lines:
#
#   "fresh start: ... the SFT-v4 soup is the base"  -> session 1 only
#   "resuming from HF: ..._step_N.pt (step N)"      -> sessions 2 and 3
#
# If session 2 or 3 says "fresh start", STOP — it would silently retrain from 0.
#
# NOTHING IS DELETED HERE. An earlier notebook cleared partial artefacts of the
# prefix before launching, which is right for a fresh start and catastrophic for
# a resume. For a multi-session run, deleting is never correct.
!cd /content/osrt && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt \
    --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --ckpt-interval 10 \
    --num-prompts 16 \
    --micro-batch 24 \
    --total-steps $TOTAL_STEPS \
    --max-gen-len $MAX_GEN_LEN \
    --seed $SEED \
    --run-tag "$RUN_TAG" \
    --kl-abort $KL_ABORT \
    --peak-lr $PEAK_LR --hra-lr $HRA_LR \
    --compile 2>&1 | tee -a /content/grpo.log

In [ ]:
# ── 4. Log digest ────────────────────────────────────────────────────
!grep -E "^step|resuming|fresh start|saved|pushed|ABORT|Error|Traceback|CUDA out of memory" \
    /content/grpo.log | tail -50

### What to watch while it runs

- **`live N/M`** — rollouts with non-zero advantage. Collapsing toward 0 means
  learning has stopped whatever reward does.
- **`cap` vs `noclose`** — different failures. `cap` is hitting `MAX_GEN_LEN`;
  `noclose` is a missing `<|/answer|>` tag. Because generation stops on the close
  tag, `cap` is a subset of `noclose`; both 40-step runs kept them identical, so
  a gap opening would be new information.
- **`kl`** — expect ~0.0024/step, so ~0.24 by step 100 and ~0.48 by 200. That is
  the prediction, not a problem. `ABORT` at 0.60 means runaway.
- **`acc`** — **ignore it.** 16 prompts, residual sd 8.13pp, and
  `corr(reward, acc) = +0.976` so reward adds nothing beyond it.
- **The printed rollouts.** Every real defect in this project was found in
  generation text, not in scalars.
- **EMA residual weight** — printed at each save. `0.99^200 ≈ 0.13`, so unlike
  the 40-step runs (0.669, two-thirds soup) the base largely washes out and the
  shadow becomes a real test of averaging.

In [ ]:
# ── 5. Panel eval — run at steps 50 / 100 / 150 / 200 ────────────────
# Scores the candidates AND the soup on the SAME 200 problems, then computes
# paired differences bootstrapping over resampled QUESTIONS. Paired because both
# see identical items, so difficulty is shared; treating the means as
# independent understates uncertainty and once turned a p=0.186 trend into a
# "significant" one.
TAG = "grpo_v6b_s7x200"
CANDIDATES = [f"{TAG}_step_{s}.pt" for s in (50, 100, 150, 200)] + \
             [f"{TAG}_final.pt", f"{TAG}_ema_final.pt"]
BASELINE = "osrt_v5_sft_v4_soup_1200_1400_1600_1800.pt"
N_PROBLEMS = 200
PROBLEM_OFFSET = 0     # 0 = development panel. 200 = untouched confirmation set.

import os, random, sys
import torch
sys.path.insert(0, "/content/osrt/src")
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from osrt.model import OSRTForCausalLM
from osrt.presets import build_config
from osrt.sft_eval import run_reasoning_eval

tok = AutoTokenizer.from_pretrained("/content/osrt/v6_tokenizer_export")
assert len(tok) == 65536, f"wrong tokenizer: {len(tok)}"
cfg = build_config(vocab_size=len(tok), real_vocab_size=len(tok),
                   bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
                   pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8)
device = torch.device("cuda")
model = OSRTForCausalLM(cfg).to(device)

def score(name):
    path = f"/content/ckpt/{name}"
    if not os.path.exists(path):
        path = hf_hub_download("HallD/osrt-v6-ckpt", name, repo_type="model")
    ck = torch.load(path, map_location=device, weights_only=True)
    miss, unexp = model.load_state_dict(ck.get("model_state_dict", ck), strict=False)
    assert not miss and not unexp, f"{name}: {miss[:3]} {unexp[:3]}"
    meta = {k: ck[k] for k in ("seed", "peak_lr", "ema_updates") if k in ck}
    del ck
    model.eval()
    if hasattr(model, "set_moe_telemetry"):
        model.set_moe_telemetry(False)
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        m = run_reasoning_eval(model, tok, device, n_problems=N_PROBLEMS,
                               max_new_tokens=512, batch_size=32,
                               repetition_penalty=1.2, return_items=True,
                               problem_offset=PROBLEM_OFFSET)
    print(f"{name:<36} acc_on {100*m['sft_eval/acc_on']:5.1f}%  "
          f"acc_off {100*m['sft_eval/acc_off']:5.1f}%  "
          f"delta {100*m['sft_eval/acc_delta_on_minus_off']:+5.1f}pp  "
          f"fmt {100*m['sft_eval/format_ok_on']:5.1f}%  "
          f"len {m['sft_eval/resp_len_on']:.0f}  {meta}", flush=True)
    return m

results = {BASELINE: score(BASELINE)}
for c in CANDIDATES:
    try:
        results[c] = score(c)
    except Exception as e:
        print(f"{c:<36} skipped ({type(e).__name__})")

rng = random.Random(0); REPS = 10000
base = results[BASELINE]["items"]
print(f"\npaired vs the soup, {REPS} reps over resampled QUESTIONS  "
      f"[(+1)/(B+1) tails]")
for c in CANDIDATES:
    if c not in results: continue
    for key in ("on", "off"):
        a, b = base[key], results[c]["items"][key]
        n = len(a); d = 100.0*(sum(a)-sum(b))/n
        ao = sum(1 for i in range(n) if a[i] and not b[i])
        bo = sum(1 for i in range(n) if b[i] and not a[i])
        reps = sorted(100.0*sum(a[i]-b[i] for i in
                      [rng.randrange(n) for _ in range(n)])/n for _ in range(REPS))
        lo, hi = reps[int(.025*REPS)], reps[int(.975*REPS)]
        nle = sum(1 for x in reps if x <= 0); nge = sum(1 for x in reps if x >= 0)
        p = min(1.0, 2*(min(nle, nge)+1)/(REPS+1))
        print(f"  acc_{key:<3} soup - {c:<34} {d:+6.2f}pp  CI [{lo:+.2f},{hi:+.2f}]  "
              f"p={p:.3f}  disc {ao}/{bo}  "
              f"{'EXCLUDES 0' if not (lo <= 0 <= hi) else 'crosses 0'}")
print("\nA CI crossing zero means 'cannot distinguish', NOT 'equal'.")

## After it finishes

**1. Harvest, do not ship the endpoint.** In SFT-v4 the final checkpoint was
measurably not the best; the wave-2 soup beat its own endpoint by 1.0pp `acc_on`
with a 4pp better delta; and seed 2's EMA beat its own θ by **+5.50pp
(p=0.023)**. Score every interval checkpoint, the final, and the EMA.

**2. Soup the late checkpoints** — `app.py::run_make_soup --prefix
grpo_v6b_s7x200 --steps 140,160,180,200` — then score the soup too.

**3. Debias the EMA** — `app.py::run_debias_ema` removes the base's residual
weight, giving the exponentially-weighted average of the trajectory alone:
`theta_avg = (e_n - d^n * theta_0) / (1 - d^n)`. At 200 steps `0.99^200 ≈ 0.13`,
so this matters much less than it did at 40 (0.669), but it is still the clean
test of averaging versus base proximity.

**4. Drift is diagnostic** — `app.py::run_ckpt_drift`. Report the HRA:base ratio;
the adapters run at 10x the base lr and moved 49x as far, so an update can look
healthy in aggregate while the expert weights barely move.

**5. Nothing here licenses a capability claim.** One seed, and the panel has been
used for selection all along. A claim needs a second seed and
`PROBLEM_OFFSET=200` with candidates declared in advance — see
`docs/specs/2026-08-10-verification-reward-prereg.md` §4b.